### Model Training

In [1]:
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

# Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Boosting models
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


# Metrics
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score, classification_report, confusion_matrix,roc_auc_score

In [2]:
df = pd.read_csv("final_dataset_2000_2022_long_year_order.csv")

# Drop unnecessary columns
df_model = df.drop(columns=["Year"])

# Group by country
df_country = df_model.groupby(['Country', 'Country Code']).mean().reset_index()

print(df_country.shape)
df_country.head()

(136, 17)


,Country,Country Code,Predictor_01 - Food supply (kcal/capita/day),Predictor_02 - Protein supply quantity (g/capita/day),Predictor_03 - Fat supply quantity (g/capita/day),Predictor_04 - GDP per capita(current US$),Predictor_05 - HDI Value,Predictor_06 - Population Growth Rate(%),Predictor_07 - Food production index,Predictor_08 - People using basic drinking water services (% of population),Predictor_09 - Prevalance of Maternal Anemia %,TV_01 - Stunting,TV_02 - Prevalence_of_Overweight,Stunting_norm,Overweight_norm,Raw_norm,Malnutrition_Index
0,Afghanistan,AFG,2102.831304,59.465652,36.482174,415.570470,0.454957,3.100613,90.753043,51.796415,34.495652,46.286957,4.752174,0.809037,0.147764,0.956801,0.629916
1,Albania,ALB,3101.297391,105.943913,101.223478,4150.750415,0.761261,-1.032496,87.361739,91.476801,24.552174,19.078261,20.047826,0.324033,0.692093,1.016127,0.673245
2,Algeria,DZA,3204.355652,88.296522,83.466087,4160.733848,0.715826,1.740564,80.800435,90.767362,34.247826,14.021739,13.521739,0.233899,0.459848,0.693747,0.437795
3,Angola,AGO,2276.012609,48.682174,50.493043,3017.657977,0.530217,3.572859,80.870870,52.388479,47.321739,37.230435,3.104348,0.647601,0.089123,0.736724,0.469183
4,Argentina,ARG,3157.732174,104.575217,113.808261,9438.910541,0.835913,0.914491,90.373913,98.210784,27.847826,8.043478,11.134783,0.127335,0.374903,0.502238,0.297926


Classification Labels

In [3]:
def categorize(index):
    if index < 0.15:
        return "Low"
    elif index < 0.30:
        return "Moderate"
    elif index < 0.40:
        return "Hotspot"
    else:
        return "Severe"

df_country["Target"] = df_country["Malnutrition_Index"].apply(categorize)

Encode Target

In [4]:
le = LabelEncoder()
df_country["Target_encoded"] = le.fit_transform(df_country["Target"])

Features and Target

In [5]:
df_country.columns

Index(['Country', 'Country Code',
       'Predictor_01 - Food supply (kcal/capita/day)',
       'Predictor_02 - Protein supply quantity (g/capita/day)',
       'Predictor_03 - Fat supply quantity (g/capita/day)',
       'Predictor_04 - GDP per capita(current US$)',
       'Predictor_05 - HDI Value', 'Predictor_06 - Population Growth Rate(%)',
       'Predictor_07 - Food production index',
       'Predictor_08 - People using basic drinking water services (% of population)',
       'Predictor_09 - Prevalance of Maternal Anemia %', 'TV_01 - Stunting',
       'TV_02 - Prevalence_of_Overweight', 'Stunting_norm', 'Overweight_norm',
       'Raw_norm', 'Malnutrition_Index', 'Target', 'Target_encoded'],
      dtype='object')

In [6]:
X = df_country.drop(columns=["Country","Country Code", "Malnutrition_Index", "Target", "Target_encoded","TV_01 - Stunting","TV_02 - Prevalence_of_Overweight","Stunting_norm","Overweight_norm","Raw_norm"])
y = df_country["Target_encoded"]

In [7]:
X.head()

,Predictor_01 - Food supply (kcal/capita/day),Predictor_02 - Protein supply quantity (g/capita/day),Predictor_03 - Fat supply quantity (g/capita/day),Predictor_04 - GDP per capita(current US$),Predictor_05 - HDI Value,Predictor_06 - Population Growth Rate(%),Predictor_07 - Food production index,Predictor_08 - People using basic drinking water services (% of population),Predictor_09 - Prevalance of Maternal Anemia %
0,2102.831304,59.465652,36.482174,415.570470,0.454957,3.100613,90.753043,51.796415,34.495652
1,3101.297391,105.943913,101.223478,4150.750415,0.761261,-1.032496,87.361739,91.476801,24.552174
2,3204.355652,88.296522,83.466087,4160.733848,0.715826,1.740564,80.800435,90.767362,34.247826
3,2276.012609,48.682174,50.493043,3017.657977,0.530217,3.572859,80.870870,52.388479,47.321739
4,3157.732174,104.575217,113.808261,9438.910541,0.835913,0.914491,90.373913,98.210784,27.847826


In [8]:
y.head()

0    3
1    3
2    3
3    3
4    2
Name: Target_encoded, dtype: int64

Preprocessing pipeline

In [9]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

K-Fold Cross validation (use 5 folds)

In [10]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Model Training with GridSearchCV(Hyper-parameter tunning) + K- FOLD

In [11]:
def evaluate(model, X, y):
    acc_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []
    roc_auc_scores = []
    
    for train_idx, test_idx in kf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        X_train = pd.DataFrame(pipeline.fit_transform(X_train),columns=X.columns)
        X_test = pd.DataFrame(pipeline.transform(X_test),columns=X.columns)
        
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)
        
        acc_scores.append(accuracy_score(y_test, pred))
        precision_scores.append(precision_score(y_test, pred, average="weighted"))
        recall_scores.append(recall_score(y_test, pred, average="weighted"))
        roc_auc_scores.append(roc_auc_score(y_test, proba, multi_class="ovr", average="weighted"))
        f1_scores.append(f1_score(y_test, pred, average="weighted"))
    
    return np.mean(acc_scores), np.mean(f1_scores), np.mean(precision_scores), np.mean(recall_scores), np.mean(roc_auc_scores)

In [12]:
def tune_model(model, param_grid):
    
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    
    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring="f1_weighted",
        n_jobs=-1
    )

    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)
    
    grid.fit(X_resampled, y_resampled)
    
    print("Best Params:", grid.best_params_)
    
    return grid.best_estimator_

Random Forest

In [13]:
rf_params = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

rf_best = tune_model(RandomForestClassifier(random_state=42), rf_params)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Best Params: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}


XGBoost

In [14]:
xgb_params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

xgb_best = tune_model(XGBClassifier(eval_metric="mlogloss"), xgb_params)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Best Params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 100, 'model__subsample': 1.0}


CatBoost

In [15]:
cat_params = {
    "model__iterations": [100, 200],
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.1]
}

cat_best = tune_model(CatBoostClassifier(verbose=0), cat_params)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Best Params: {'model__depth': 8, 'model__iterations': 200, 'model__learning_rate': 0.1}


Decision tree

In [16]:
dt_params = {
    "model__max_depth": [3, 5, 10, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

dt_best = tune_model(DecisionTreeClassifier(random_state=42), dt_params)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Best Params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2}


Gradient Boosting

In [17]:
gb_params = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.01, 0.1],
    "model__max_depth": [3, 5]
}

gb_best = tune_model(GradientBoostingClassifier(), gb_params)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}


KNN

In [18]:
knn_params = {
    "model__n_neighbors": [3, 5, 7, 9],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]  # Manhattan vs Euclidean
}

knn_best = tune_model(KNeighborsClassifier(), knn_params)

Best Params: {'model__n_neighbors': 3, 'model__p': 2, 'model__weights': 'distance'}


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Logistic Regression

In [19]:
lr_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs", "liblinear"]
}

lr_best = tune_model(LogisticRegression(max_iter=1000), lr_params)

Best Params: {'model__C': 10, 'model__solver': 'lbfgs'}


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


SVM

In [20]:
svm_params = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"]
}

svm_best = tune_model(SVC(probability=True), svm_params)

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Best Params: {'model__C': 10, 'model__gamma': 'scale', 'model__kernel': 'rbf'}


Results

In [21]:
models = {
    "RF": rf_best,
    "XGB": xgb_best,
    "CatBoost": cat_best,
    "DT": dt_best,
    "GB": gb_best,
    "KNN": knn_best,
    "LR": lr_best,
    "SVM": svm_best
}

results = []

for name, model in models.items():
    acc, f1, prec, rec, auc = evaluate(model, X, y)
    
    results.append([name, acc, prec, rec, f1, auc])

results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"
])

print(results_df.sort_values(by="F1 Score", ascending=False))

/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{m

      Model  Accuracy  Precision    Recall  F1 Score   ROC AUC
0        RF  0.573545   0.563378  0.573545  0.548134  0.775982
5       KNN  0.529894   0.516067  0.529894  0.519833  0.739291
2  CatBoost  0.529365   0.517484  0.529365  0.505726  0.779340
6        LR  0.529630   0.512945  0.529630  0.499387  0.769831
7       SVM  0.499471   0.530239  0.499471  0.486558  0.730893
1       XGB  0.507143   0.473211  0.507143  0.482086  0.752351
4        GB  0.471164   0.467048  0.471164  0.460121  0.741682
3        DT  0.478571   0.462044  0.478571  0.459584  0.636273


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
results_df.to_csv("model_comparison_Classification_results.csv", index=False)

In [23]:
# Filter only selected models
selected_models = ["LR", "SVM", "RF", "XGB", "CatBoost"]

final_table = results_df[results_df["Model"].isin(selected_models)]

# Round to 2 decimal places
final_table = final_table.round(2)

# Sort by F1 Score (best model on top)
final_table = final_table.sort_values(by="F1 Score", ascending=False)

print(final_table)

      Model  Accuracy  Precision  Recall  F1 Score  ROC AUC
0        RF      0.57       0.56    0.57      0.55     0.78
2  CatBoost      0.53       0.52    0.53      0.51     0.78
6        LR      0.53       0.51    0.53      0.50     0.77
7       SVM      0.50       0.53    0.50      0.49     0.73
1       XGB      0.51       0.47    0.51      0.48     0.75
